In [ ]:
import os, sys
os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'
os.environ['HF_HUB_DISABLE_XET'] = '1'

os.system('pip uninstall unsloth unsloth_zoo -y -q')
os.system('pip install --no-cache-dir --upgrade "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git" -q')
os.system('pip install --no-cache-dir --upgrade "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git" -q')
os.system('pip install "transformers==4.46.3" "trl>=0.18.2,<=0.24.0,!=0.19.0" -q')
os.system('pip install "datasets>=3.4.1,<4.4.0" "torchao>=0.13.0" -q')
os.system('pip install cut_cross_entropy hf_transfer msgspec tyro gradio -q')

for mod in list(sys.modules):
    if mod.startswith(("unsloth", "unsloth_zoo")):
        del sys.modules[mod]

print("✅ Packages installed.")


In [ ]:
import json, random, re, os
from datetime import datetime, timedelta

_RAW_PRODUCTS = {
    "dresses": [
        {"name": "Luna",   "colors": ["Black", "White", "Red", "Blue"],         "sizes": ["XS","S","M","L","XL"],         "price": 49.99},
        {"name": "Stella", "colors": ["Navy", "Burgundy", "Green"],             "sizes": ["S","M","L","XL"],              "price": 59.99},
        {"name": "Aurora", "colors": ["Pink", "Lavender", "Mint"],              "sizes": ["XS","S","M","L"],              "price": 54.99},
        {"name": "Nicol",  "colors": ["Black", "Charcoal", "Beige"],            "sizes": ["S","M","L","XL","XXL"],        "price": 64.99},
        {"name": "Summer", "colors": ["Yellow", "Coral", "Turquoise"],          "sizes": ["XS","S","M","L"],              "price": 44.99},
        {"name": "Verona", "colors": ["Black", "White", "Olive"],               "sizes": ["XS","S","M","L","XL"],         "price": 52.99},
        {"name": "Sofia",  "colors": ["Rose", "Sage", "Ivory"],                 "sizes": ["S","M","L"],                   "price": 57.99},
    ],
    "tops": [
        {"name": "Vega",   "colors": ["White", "Blue", "Black", "Grey"],        "sizes": ["XS","S","M","L","XL"],         "price": 29.99},
        {"name": "Cloud",  "colors": ["Cream", "Pink", "Lilac"],                "sizes": ["S","M","L"],                   "price": 34.99},
        {"name": "Silk",   "colors": ["Champagne", "Pearl", "Ivory"],           "sizes": ["XS","S","M","L"],              "price": 39.99},
        {"name": "Cotton", "colors": ["White", "Black", "Navy", "Olive"],       "sizes": ["S","M","L","XL"],              "price": 24.99},
        {"name": "Breeze", "colors": ["Sky Blue", "Mint", "Peach"],             "sizes": ["XS","S","M","L"],              "price": 27.99},
        {"name": "Urban",  "colors": ["Black", "White", "Grey", "Red"],         "sizes": ["S","M","L","XL","XXL"],        "price": 32.99},
    ],
    "jeans": [
        {"name": "Classic",  "colors": ["Blue", "Black", "Grey"],               "sizes": ["26","28","30","32","34"],      "price": 69.99},
        {"name": "Slim",     "colors": ["Dark Blue", "Black"],                  "sizes": ["26","28","30","32"],           "price": 74.99},
        {"name": "Straight", "colors": ["Blue", "Light Blue", "Black"],         "sizes": ["28","30","32","34","36"],      "price": 64.99},
        {"name": "Relaxed",  "colors": ["Blue", "Black", "Sand"],               "sizes": ["28","30","32","34","36"],      "price": 67.99},
    ],
    "skirts": [
        {"name": "Star",  "colors": ["Black", "Red", "Navy"],                   "sizes": ["XS","S","M","L"],              "price": 39.99},
        {"name": "Midi",  "colors": ["Beige", "Brown", "Black"],                "sizes": ["S","M","L","XL"],              "price": 44.99},
        {"name": "Pleat", "colors": ["Navy", "Burgundy", "Forest"],             "sizes": ["XS","S","M","L"],              "price": 49.99},
        {"name": "Wrap",  "colors": ["Floral", "Stripe", "Solid Black"],        "sizes": ["XS","S","M","L","XL"],         "price": 42.99},
    ],
    "shoes": [
        {"name": "Comfort", "colors": ["Black", "White", "Beige"],              "sizes": ["36","37","38","39","40"],      "price": 89.99},
        {"name": "Sport",   "colors": ["White", "Black", "Grey", "Blue"],       "sizes": ["36","37","38","39","40","41"], "price": 79.99},
        {"name": "Elegant", "colors": ["Black", "Nude", "Silver"],              "sizes": ["36","37","38","39","40"],      "price": 99.99},
        {"name": "Casual",  "colors": ["White", "Beige", "Brown"],              "sizes": ["36","37","38","39","40","41"], "price": 69.99},
    ],
}

PRODUCTS = {}
PRODUCTS_LIST = []
_pid = 1
for cat, items in _RAW_PRODUCTS.items():
    PRODUCTS[cat] = []
    for p in items:
        p_with_id = {"product_id": _pid, "category": cat, **p}
        PRODUCTS[cat].append(p_with_id)
        PRODUCTS_LIST.append(p_with_id)
        _pid += 1

def find_product_by_id(product_id):
    for p in PRODUCTS_LIST:
        if p["product_id"] == int(product_id):
            return p
    return None

def find_product(product_name, category=None):
    name_lc = (product_name or "").lower()
    cats = [category] if category else list(PRODUCTS.keys())
    for cat in cats:
        for p in PRODUCTS[cat]:
            if p["name"].lower() == name_lc:
                return cat, p
    return None, None

_CATEGORY_SINGULAR = {"dresses": "dress", "tops": "top", "jeans": "jeans",
                      "skirts": "skirt", "shoes": "shoes"}
def category_singular(cat):
    return _CATEGORY_SINGULAR.get(cat, cat or "")

_DEFAULT_CUSTOMERS = [
    {"customer_id": 1, "name": "Anna Smith",   "phone": "+380501234567", "address": "Kyiv, НП #12"},
    {"customer_id": 2, "name": "Maria Garcia", "phone": "+380671234567", "address": "Lviv, НП #5"},
    {"customer_id": 3, "name": "Olena Tkach",  "phone": "+380631234567", "address": "Odesa, НП #8"},
    {"customer_id": 4, "name": "John Doe",     "phone": "+380991234567", "address": "Kharkiv, НП #3"},
]

def _make_default_orders():
    raw = [
        # customer_id, product_name, color, size, status, date
        (1, "Luna",    "Black", "M",  "delivered",        "2026-04-01"),
        (1, "Vega",    "White", "S",  "shipped",          "2026-04-08"),
        (2, "Aurora",  "Pink",  "S",  "processing",       "2026-04-09"),
        (2, "Classic", "Blue",  "30", "delivered",        "2026-03-25"),
        (3, "Stella",  "Navy",  "M",  "out_for_delivery", "2026-04-10"),
        (4, "Comfort", "Black", "40", "shipped",          "2026-04-07"),
        (4, "Slim",    "Black", "32", "processing",       "2026-04-10"),
    ]
    orders = {}
    oid = 1001
    for cid, pname, color, size, status, date in raw:
        _, prod = find_product(pname)
        if not prod:
            continue
        orders[oid] = {
            "order_id":   oid,
            "customer_id": cid,
            "product_id": prod["product_id"],
            "product_name": prod["name"],           
            "category":   prod["category"],          
            "color":      color,
            "size":       size,
            "price":      prod["price"],             
            "status":     status,
            "date":       date,
        }
        oid += 10
    return orders


DB_FILE = "/kaggle/working/demo_db.json"
DB_SCHEMA_VERSION = 2  

def _load_db():
    if os.path.exists(DB_FILE):
        try:
            with open(DB_FILE, "r", encoding="utf-8") as f:
                data = json.load(f)
            if data.get("schema_version") != DB_SCHEMA_VERSION:
                print(f"⚠️ Old DB schema (v{data.get('schema_version','?')}), reinitialising with v{DB_SCHEMA_VERSION}")
                raise ValueError("schema mismatch")
            customers = data.get("customers", [])
            orders    = {int(k): v for k, v in data.get("orders", {}).items()}
            returns   = data.get("returns",   [])
            print(f"✅ DB loaded (v{DB_SCHEMA_VERSION}): {len(customers)} customers, {len(orders)} orders, {len(returns)} returns")
            return customers, orders, returns
        except Exception as e:
            print(f"⚠️ Could not load DB ({e}), using defaults")
    print(f"🆕 Initialising fresh DB (v{DB_SCHEMA_VERSION})")
    return (
        [dict(c) for c in _DEFAULT_CUSTOMERS],
        _make_default_orders(),
        [],
    )

def save_db():
    try:
        with open(DB_FILE, "w", encoding="utf-8") as f:
            json.dump({
                "schema_version": DB_SCHEMA_VERSION,
                "customers": CUSTOMERS,
                "orders":    {str(k): v for k, v in ALL_ORDERS.items()},
                "returns":   RETURNS,
            }, f, ensure_ascii=False, indent=2)
    except Exception as e:
        print(f"⚠️ save_db failed: {e}")


CUSTOMERS, ALL_ORDERS, RETURNS = _load_db()


def find_customer_by_id(customer_id):
    for c in CUSTOMERS:
        if c["customer_id"] == int(customer_id):
            return c
    return None

def find_customer_by_name(name):
    if not name:
        return None
    name_lc = name.lower()
    for c in CUSTOMERS:
        if c["name"].lower() == name_lc:
            return c
    return None

def get_customer_orders(customer_id):
    return [o for o in ALL_ORDERS.values() if o.get("customer_id") == int(customer_id)]

def get_customer_names():
    return sorted(c["name"] for c in CUSTOMERS)

def _next_customer_id():
    return (max((c["customer_id"] for c in CUSTOMERS), default=0)) + 1

def _next_return_id():
    return (max((int(r["return_id"]) for r in RETURNS if str(r.get("return_id","")).isdigit()), default=0)) + 1


SESSION = {
    "Qwen 2.5 1.5B": {"customer_id": None, "drafts": [], "payment": None, "delivery": None},
    "Llama 3.2 1B":  {"customer_id": None, "drafts": [], "payment": None, "delivery": None},
}

def reset_session(model_label, customer=None):
    cust = find_customer_by_name(customer) if customer else None
    SESSION[model_label] = {
        "customer_id": cust["customer_id"] if cust else None,
        "drafts":      [],
        "payment":     None,
        "delivery":    None,
    }


def fn_check_product_availability(session, product_name, color=None, size=None, **_):
    cat, prod = find_product(product_name)
    if not prod:
        return {"available": False, "stock_quantity": 0, "error": f"Unknown product: {product_name}"}
    color_ok = (color is None) or (color in prod["colors"])
    size_ok  = (size is None)  or (size  in prod["sizes"])
    available = color_ok and size_ok
    return {
        "available": available,
        "stock_quantity": random.randint(3, 20) if available else 0,
        "price": prod["price"],
        "available_colors": prod["colors"],
        "available_sizes":  prod["sizes"],
    }

def fn_search_products(session, category=None, color=None, **_):
    if category and category not in PRODUCTS:
        for c in PRODUCTS:
            if category.lower() in c.lower():
                category = c
                break
    cats = [category] if category in PRODUCTS else list(PRODUCTS.keys())
    out = []
    for cat in cats:
        for p in PRODUCTS[cat]:
            if color and color not in p["colors"]:
                continue
            out.append({"name": p["name"], "category": cat,
                        "colors": p["colors"], "sizes": p["sizes"], "price": p["price"]})
    return {"category": category, "results": out[:10], "total": len(out)}

def fn_create_order(session, product_name, color, size, quantity=1, **_):
    cat, prod = find_product(product_name)
    if not prod:
        return {"success": False, "error": f"Unknown product: {product_name}"}
    order_id = random.randint(2000, 9999)
    order = {
        "order_id":    order_id,
        "customer_id": session.get("customer_id"),     
        "product_id":  prod["product_id"],
        "product_name": prod["name"],
        "category":    cat or "—",
        "color":       color,
        "size":        size,
        "quantity":    quantity,
        "price":       prod["price"],
        "status":      "draft",
        "date":        datetime.now().strftime("%Y-%m-%d"),
    }
    session["drafts"].append(order)
    ALL_ORDERS[order_id] = order
    return {"order_id": order_id, "status": "draft",
            "product": f"{product_name} {category_singular(cat) if cat else ''}".strip(),
            "size": size, "color": color}

def fn_confirm_payment(session, payment_method, **_):
    session["payment"] = payment_method
    for d in session["drafts"]:
        d["payment_method"] = payment_method
    return {"status": "paid", "transaction_id": f"TXN_{random.randint(1000, 9999)}"}

def fn_add_delivery_details(session, name=None, phone=None, city=None, post_office=None, **_):
    delivery = {"name": name, "phone": phone, "city": city, "post_office": post_office}
    session["delivery"] = delivery
    for d in session["drafts"]:
        d["delivery"] = delivery
    return {"delivery_id": f"DEL_{random.randint(100, 999)}"}

def fn_confirm_order(session, **_):
    if not session.get("drafts"):
        return {"error": "no_drafts", "message": "No items in cart to confirm."}

    confirmed = []
    delivery = session.get("delivery") or {}
    delivery_name = delivery.get("name")

    cust = find_customer_by_id(session["customer_id"]) if session.get("customer_id") else None
    if not cust and delivery_name:
        address_parts = []
        if delivery.get("city"):        address_parts.append(delivery["city"])
        if delivery.get("post_office"): address_parts.append(f"НП #{delivery['post_office']}")
        cust = {
            "customer_id": _next_customer_id(),
            "name":        delivery_name,
            "phone":       delivery.get("phone") or "—",
            "address":     ", ".join(address_parts) if address_parts else "—",
        }
        CUSTOMERS.append(cust)
        session["customer_id"] = cust["customer_id"]
    elif cust and delivery_name:
        cust["phone"] = delivery.get("phone") or cust.get("phone", "—")
        address_parts = []
        if delivery.get("city"):        address_parts.append(delivery["city"])
        if delivery.get("post_office"): address_parts.append(f"НП #{delivery['post_office']}")
        if address_parts:
            cust["address"] = ", ".join(address_parts)

    cid = cust["customer_id"] if cust else None

    for d in session["drafts"]:
        d["status"] = "confirmed"
        d["customer_id"] = cid
        if delivery:
            d["delivery"] = delivery
        ALL_ORDERS[d["order_id"]] = d
        confirmed.append({"order_id": d["order_id"], "status": "confirmed"})

    session["drafts"] = []
    save_db()
    return {"confirmed_orders": confirmed}

def fn_track_order(session, order_id, **_):
    try:
        oid = int(order_id)
    except (TypeError, ValueError):
        return {"error": f"Invalid order id: {order_id}"}
    if oid not in ALL_ORDERS:
        return {"error": f"Order {oid} not found"}
    o = ALL_ORDERS[oid]
    eta = (datetime.now() + timedelta(days=random.randint(1, 3))).strftime("%Y-%m-%d")
    return {"order_id": oid, "status": o.get("status", "processing"),
            "product": f"{o['product_name']} {category_singular(o.get('category',''))}".strip(),
            "color": o.get("color"), "size": o.get("size"), "estimated_delivery": eta}

def fn_cancel_order(session, order_id, **_):
    try:
        oid = int(order_id)
    except (TypeError, ValueError):
        return {"cancelled": False, "error": f"Invalid order id: {order_id}"}
    if oid not in ALL_ORDERS:
        return {"cancelled": False, "error": f"Order {oid} not found"}
    o = ALL_ORDERS[oid]
    if o.get("status") in ("delivered", "cancelled"):
        return {"cancelled": False, "error": f"Cannot cancel — already {o['status']}"}
    o["status"] = "cancelled"
    save_db()
    return {"cancelled": True, "order_id": oid, "refund_amount": o.get("price", 0),
            "message": f"Order {oid} cancelled successfully"}

def fn_update_order_item(session, order_id, field, new_value, **_):
    try:
        oid = int(order_id)
    except (TypeError, ValueError):
        return {"updated": False, "error": f"Invalid order id: {order_id}"}
    if oid not in ALL_ORDERS:
        return {"updated": False, "error": f"Order {oid} not found"}
    o = ALL_ORDERS[oid]
    if o.get("status") in ("shipped", "delivered", "out_for_delivery", "cancelled"):
        return {"updated": False, "message": f"Order already in {o['status']}. Cannot update."}
    if field not in ("size", "color"):
        return {"updated": False, "error": f"Cannot update field '{field}'. Allowed: size, color."}
    o[field] = new_value
    save_db()
    return {"updated": True, "order_id": oid, "new_status": o.get("status", "draft"),
            "field": field, "new_value": new_value}

def fn_create_return_request(session, order_id, reason, **_):
    try:
        oid = int(order_id)
    except (TypeError, ValueError):
        return {"status": "rejected", "error": f"Invalid order id: {order_id}"}
    if oid not in ALL_ORDERS:
        return {"status": "rejected", "error": f"Order {oid} not found"}
    o = ALL_ORDERS[oid]
    rid = _next_return_id()
    RETURNS.append({
        "return_id":     rid,
        "order_id":      oid,
        "reason":        reason,
        "status":        "approved",
        "refund_amount": o.get("price", 0),
    })
    o["status"] = "returned"
    save_db()
    return {"return_id": rid, "status": "approved", "refund_amount": o.get("price", 0),
            "instructions": "Drop off at any Nova Poshta branch within 14 days."}

def fn_get_current_order_context(session, **_):
    active = []
    cid = session.get("customer_id")
    if cid is not None:
        for o in get_customer_orders(cid):
            if o.get("status") not in ("delivered", "cancelled", "returned"):
                active.append({
                    "order_id": o["order_id"],
                    "product": f"{o['product_name']} {category_singular(o.get('category',''))}".strip(),
                    "color": o.get("color"), "size": o.get("size"),
                    "status": o.get("status", "processing"),
                })
    for d in session.get("drafts", []):
        active.append({
            "order_id": d["order_id"],
            "product":  f"{d['product_name']} {category_singular(d.get('category',''))}".strip(),
            "color":    d.get("color"),
            "size":     d.get("size"),
            "status":   d.get("status", "draft"),
        })
    return {"active_orders": active}

def fn_get_order_history(session, limit=10, **_):
    cid = session.get("customer_id")
    history = []
    if cid is not None:
        for o in get_customer_orders(cid):
            history.append({
                "order_id": o["order_id"],
                "product":  f"{o['product_name']} {category_singular(o.get('category',''))}".strip(),
                "color":    o.get("color"),
                "size":     o.get("size"),
                "status":   o.get("status"),
                "date":     o.get("date"),
            })
    return {"orders": history[: int(limit) if limit else 10]}


FUNCTIONS = {
    "check_product_availability": fn_check_product_availability,
    "search_products":             fn_search_products,
    "create_order":                fn_create_order,
    "place_order":                 fn_create_order,
    "confirm_payment":             fn_confirm_payment,
    "add_delivery_details":        fn_add_delivery_details,
    "confirm_order":               fn_confirm_order,
    "track_order":                 fn_track_order,
    "cancel_order":                fn_cancel_order,
    "update_order_item":           fn_update_order_item,
    "create_return_request":       fn_create_return_request,
    "get_current_order_context":   fn_get_current_order_context,
    "get_order_history":           fn_get_order_history,
}


def _split_name(full_name):
    parts = (full_name or "").strip().split(" ", 1)
    return parts[0], parts[1] if len(parts) > 1 else ""

def _split_address(address):
    if not address or address == "—":
        return "—", "—"
    parts = address.split(", ", 1)
    return parts[0], parts[1] if len(parts) > 1 else "—"


def get_db_view(customer_name=None):
    BG_HEAD, BG_BODY, BG_ALT = "#f5efe6", "#fbf8f3", "#f0e9dc"
    BORDER, TEXT, MUTED = "#d9cfbe", "#3a352e", "#8a8074"
    status_bg = {
        "delivered": "#e3f1e1", "processing": "#fdf3d4", "shipped": "#e0eaf6",
        "out_for_delivery": "#dbe9d4", "cancelled": "#f4dcdc", "returned": "#ece0f1",
        "draft": "#f7f0e0", "confirmed": "#e6efd9",
    }

    html = f"<div style='font-family:Inter,system-ui,sans-serif;font-size:13px;color:{TEXT}'>"

    selected_cust = find_customer_by_name(customer_name) if customer_name and customer_name != "New Customer" else None

    if customer_name == "New Customer":
        view_orders = []
        html += f"<p style='color:{MUTED};margin:4px 0 10px'>New customer — no orders yet</p>"
    elif selected_cust:
        view_orders = sorted(get_customer_orders(selected_cust["customer_id"]),
                             key=lambda r: r["order_id"], reverse=True)
        html += f"<p style='color:{MUTED};margin:4px 0 10px'>Records for: <b style='color:{TEXT}'>{customer_name}</b></p>"
    else:
        view_orders = sorted(ALL_ORDERS.values(), key=lambda r: r["order_id"], reverse=True)

    html += f"<h3 style='color:{TEXT};margin:8px 0 6px;font-weight:600'>Customers</h3>"
    html += f"<table style='width:100%;border-collapse:collapse;margin-bottom:14px;background:{BG_BODY}'>"
    html += f"<tr style='background:{BG_HEAD};color:{TEXT};font-weight:600'>"
    for col in ["ID", "First Name", "Last Name", "Phone", "City", "Post Office", "Orders"]:
        html += f"<th style='padding:8px;border:1px solid {BORDER};text-align:left'>{col}</th>"
    html += "</tr>"
    display_customers = [selected_cust] if selected_cust else CUSTOMERS
    for i, cust in enumerate(display_customers):
        first, last = _split_name(cust["name"])
        city, post  = _split_address(cust.get("address") or "—")
        bg = BG_BODY if i % 2 == 0 else BG_ALT
        n_orders = len(get_customer_orders(cust["customer_id"]))
        html += f"<tr style='background:{bg}'>"
        for val in [cust["customer_id"], first, last, cust.get("phone") or "—", city, post, n_orders]:
            html += f"<td style='padding:6px;border:1px solid {BORDER};color:{TEXT}'>{val}</td>"
        html += "</tr>"
    if not display_customers:
        html += f"<tr><td colspan='7' style='padding:8px;color:{MUTED};text-align:center'>No customers</td></tr>"
    html += "</table>"

    html += f"<h3 style='color:{TEXT};margin:8px 0 6px;font-weight:600'>Orders</h3>"
    html += f"<table style='width:100%;border-collapse:collapse;margin-bottom:14px;background:{BG_BODY}'>"
    html += f"<tr style='background:{BG_HEAD};color:{TEXT};font-weight:600'>"
    for col in ["ID", "Product", "Color", "Size", "Price", "Status", "Date", "Customer ID"]:
        html += f"<th style='padding:8px;border:1px solid {BORDER};text-align:left'>{col}</th>"
    html += "</tr>"
    if view_orders:
        for o in view_orders:
            bg = status_bg.get(o.get("status",""), BG_BODY)
            html += f"<tr style='background:{bg}'>"
            prod = f"{o.get('product_name','')} {category_singular(o.get('category',''))}".strip()
            cid_display = o.get("customer_id") if o.get("customer_id") is not None else "Guest"
            for val in [o.get("order_id","—"), prod, o.get("color","—"), o.get("size","—"),
                        f"${o.get('price','—')}", o.get("status","—"), o.get("date","—"),
                        cid_display]:
                html += f"<td style='padding:6px;border:1px solid {BORDER};color:{TEXT}'>{val}</td>"
            html += "</tr>"
    else:
        html += f"<tr><td colspan='8' style='padding:8px;color:{MUTED};text-align:center'>No orders</td></tr>"
    html += "</table>"

    html += f"<h3 style='color:{TEXT};margin:8px 0 6px;font-weight:600'>Catalog</h3>"
    html += f"<table style='width:100%;border-collapse:collapse;margin-bottom:14px;background:{BG_BODY}'>"
    html += f"<tr style='background:{BG_HEAD};color:{TEXT};font-weight:600'>"
    for col in ["ID", "Category", "Name", "Colors", "Sizes", "Price"]:
        html += f"<th style='padding:8px;border:1px solid {BORDER};text-align:left'>{col}</th>"
    html += "</tr>"
    for i, p in enumerate(PRODUCTS_LIST):
        bg = BG_BODY if i % 2 == 0 else BG_ALT
        html += f"<tr style='background:{bg}'>"
        for val in [p["product_id"], p["category"], p["name"],
                    ", ".join(p["colors"]), ", ".join(p["sizes"]), f"${p['price']}"]:
            html += f"<td style='padding:6px;border:1px solid {BORDER};color:{TEXT}'>{val}</td>"
        html += "</tr>"
    html += "</table>"

    html += f"<h3 style='color:{TEXT};margin:8px 0 6px;font-weight:600'>Returns</h3>"
    if RETURNS:
        html += f"<table style='width:100%;border-collapse:collapse;background:{BG_BODY}'>"
        html += f"<tr style='background:{BG_HEAD};color:{TEXT};font-weight:600'>"
        for col in ["Return ID", "Order ID", "Reason", "Status", "Refund"]:
            html += f"<th style='padding:8px;border:1px solid {BORDER};text-align:left'>{col}</th>"
        html += "</tr>"
        for r in RETURNS[-10:]:
            html += f"<tr style='background:{BG_BODY}'>"
            for val in [r["return_id"], r["order_id"], r["reason"],
                        r.get("status","approved"), f"${r['refund_amount']}"]:
                html += f"<td style='padding:6px;border:1px solid {BORDER};color:{TEXT}'>{val}</td>"
            html += "</tr>"
        html += "</table>"
    else:
        html += f"<p style='color:{MUTED};margin:4px 0 12px'>No returns yet</p>"

    html += "</div>"
    return html


print(f"Catalog: {len(PRODUCTS_LIST)} products in {len(PRODUCTS)} categories (product_id 1..{len(PRODUCTS_LIST)})")
print(f"Customers: {len(CUSTOMERS)} | Orders: {len(ALL_ORDERS)} | Returns: {len(RETURNS)} | DB: {DB_FILE}")


In [ ]:
import unsloth  
import torch
from unsloth import FastLanguageModel
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

SYSTEM_PROMPT = (
    "You are an AI shopping assistant for an online clothing store. "
    "You can help customers: browse products, check availability, place orders, "
    "confirm payment, add delivery details, track orders, cancel orders, "
    "update orders, and process returns. "
    "Always call check_product_availability before creating any order. "
    "If the customer does not specify color or size, ask for them before proceeding. "
    "Keep responses short and clear."
)

BITEXT_SYSTEM = (
    "You are a helpful customer support assistant for an online clothing store. "
    "Help customers with their questions and issues politely and professionally."
)

MODELS = {
    "Qwen 2.5 1.5B": "manoilokate/qwen2.5-1.5b-ecommerce-v2",
    "Llama 3.2 1B":  "manoilokate/llama-3.2-1b-ecommerce-v3",
}
_loaded = {}

def load_model(label):
    key = MODELS[label]
    if key in _loaded:
        return _loaded[key]
    print(f"Loading {label} → {key}...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=key,
        max_seq_length=2048,
        load_in_4bit=True,
        token=HF_TOKEN,
    )
    FastLanguageModel.for_inference(model)
    _loaded[key] = (model, tokenizer)
    print(f"{label} loaded.")
    return _loaded[key]

print("Model loader ready.")
print(f"  Qwen  → {MODELS['Qwen 2.5 1.5B']}")
print(f"  Llama → {MODELS['Llama 3.2 1B']}")
print(f"  SYSTEM_PROMPT (training-exact): {SYSTEM_PROMPT[:80]}...")


In [ ]:
import re, json, time, torch

def parse_tool_call(text):
    m = re.search(r"<tool_call>(.*?)</tool_call>", text, re.DOTALL)
    if not m:
        return None
    raw = m.group(1).strip()
    try:
        data = json.loads(raw)
    except Exception:
        m2 = re.search(r"\{.*\}", raw, re.DOTALL)
        if not m2:
            return None
        try:
            data = json.loads(m2.group(0))
        except Exception:
            return None
    return data.get("name"), data.get("arguments", {}) or {}

def strip_tool_call(text):
    return re.sub(r"<tool_call>.*?</tool_call>", "", text, flags=re.DOTALL).strip()

def clean_response(text):
    return re.sub(r"\{\{[^}]+\}\}", "—", text or "").strip()


SHOPPING_KEYWORDS = re.compile(
    r"\b(order|buy|purchase|cart|cancel|track|return|refund|delivery|"
    r"dress|top|jeans|skirt|shoes|"
    r"luna|stella|aurora|nicol|summer|verona|sofia|"
    r"vega|cloud|silk|cotton|breeze|urban|"
    r"classic|slim|straight|relaxed|"
    r"star|midi|pleat|wrap|"
    r"comfort|sport|elegant|casual|"
    r"size|color|stock|available|availability)\b",
    re.IGNORECASE,
)
SUPPORT_KEYWORDS = re.compile(
    r"\b(policy|policies|hours|contact|support|email|phone|"
    r"how (do|can|long|much) i|what (is|are) your|"
    r"do you (offer|have|accept)|where (are|is) (you|your)|"
    r"shipping (time|cost|fee)|business hours|customer service|"
    r"declined|invoice|account|sign[- ]?up|register|password)\b",
    re.IGNORECASE,
)
GREETING_RE = re.compile(
    r"^\s*(hi+|hello+|hey+|good\s+(morning|afternoon|evening|day)|"
    r"greetings|howdy|sup|what'?s?\s+up|yo+|hiya)\s*[!?.]*\s*$",
    re.IGNORECASE,
)
ORDER_ID_RE = re.compile(r"#?\b\d{4}\b")

CONFIRM_RE = re.compile(
    r"^\s*(yes|yeah|yep|sure|ok|okay|go ahead|do it|place|create|confirm|proceed|order it|"
    r"let'?s? do it|yes please|please do|sounds good|great|perfect)\b",
    re.IGNORECASE,
)
CARD_RE = re.compile(r"\b(card|credit|debit|visa|mastercard)\b", re.IGNORECASE)
CASH_RE = re.compile(r"\b(cash|cod|on delivery)\b", re.IGNORECASE)

_PHONE_RE = re.compile(r"(\+?\d[\d\s\-]{8,14}\d)")
_PO_RE    = re.compile(r"(?:post\s*office|post|відділення|нп|np|#)\s*(\d+)", re.IGNORECASE)
_CITY_RE  = re.compile(
    r"\b(kyiv|lviv|odesa|kharkiv|dnipro|zaporizhzhia|vinnytsia|poltava|"
    r"cherkasy|sumy|mykolaiv|kherson|rivne|lutsk|ivano.frankivsk|ternopil|"
    r"khmelnytskyi|chernivtsi|zhytomyr|chernihiv|kropyvnytskyi)\b",
    re.IGNORECASE,
)


def _has_active_drafts(model_label):
    return bool(SESSION.get(model_label, {}).get("drafts"))

def pick_system_prompt(user_message, model_hist, model_label=None):
    msg = user_message or ""
    if GREETING_RE.match(msg) and not model_hist:
        return BITEXT_SYSTEM, "💬 support"
    if model_label and _has_active_drafts(model_label):
        return SYSTEM_PROMPT, "🛒 shopping"
    if not GREETING_RE.match(msg):
        for turn in model_hist:
            if turn.get("role") == "assistant" and "<tool_call>" in (turn.get("content") or ""):
                return SYSTEM_PROMPT, "🛒 shopping"
    if ORDER_ID_RE.search(msg):
        return SYSTEM_PROMPT, "🛒 shopping"
    has_shopping = bool(SHOPPING_KEYWORDS.search(msg))
    has_support  = bool(SUPPORT_KEYWORDS.search(msg))
    if has_shopping:
        return SYSTEM_PROMPT, "🛒 shopping"
    if has_support or GREETING_RE.match(msg):
        return BITEXT_SYSTEM, "💬 support"
    if len(msg.split()) <= 5:
        return BITEXT_SYSTEM, "💬 support"
    return SYSTEM_PROMPT, "🛒 shopping"


def state_machine_call(msg, session):
    drafts  = session.get("drafts", [])
    payment = session.get("payment")
    checked = session.get("last_checked")

    if checked and not drafts and CONFIRM_RE.match(msg):
        return "create_order", {
            "product_name": checked["product_name"],
            "color":        checked["color"],
            "size":         checked["size"],
        }
    if drafts and not payment:
        if CARD_RE.search(msg):
            return "confirm_payment", {"payment_method": "card"}
        if CASH_RE.search(msg):
            return "confirm_payment", {"payment_method": "cash"}
    if drafts and payment and not session.get("delivery"):
        phone_m = _PHONE_RE.search(msg)
        po_m    = _PO_RE.search(msg)
        city_m  = _CITY_RE.search(msg)
        if phone_m or (city_m and po_m):
            name = None
            if phone_m:
                before = msg[:phone_m.start()].strip().rstrip(",").strip()
                if before:
                    name = before
            return "add_delivery_details", {
                "name":        name,
                "phone":       phone_m.group(1).strip() if phone_m else None,
                "city":        city_m.group(1).capitalize() if city_m else None,
                "post_office": po_m.group(1) if po_m else None,
            }
    if drafts and payment and session.get("delivery") and CONFIRM_RE.match(msg):
        return "confirm_order", {}
    return None


def format_tool_result(name, args, result, session=None):
    if name == "check_product_availability":
        if result.get("available"):
            return (f"✅ **{args.get('product_name')}** in {args.get('color','—')} size {args.get('size','—')} "
                    f"is available — **${result.get('price','?')}**.\n\nReady to proceed with the order?")
        return (f"❌ {args.get('product_name')} in {args.get('color','—')} size {args.get('size','—')} "
                f"is not available. Would you like a different color or size?")
    if name == "search_products":
        items = result.get("results", [])
        if not items:
            return f"No products found in {result.get('category','—')}."
        lines_ = [f"Here's what we have in **{result.get('category','products')}**:"]
        for p in items[:6]:
            lines_.append(f"• **{p['name']}** — ${p['price']} | {', '.join(p['colors'])} | {', '.join(p['sizes'])}")
        lines_.append("\nWhich one would you like?")
        return "\n".join(lines_)
    if name in ("create_order", "place_order"):
        drafts = session.get("drafts", []) if session else []
        total  = sum(d.get("price", 0) for d in drafts)
        summary = (f"📝 Draft **#{result.get('order_id')}** — "
                   f"{result.get('product','')} ({result.get('color','')}, size {result.get('size','')}).")
        if len(drafts) > 1:
            draft_lines = [f"• #{d['order_id']} — {d['product_name']} ({d.get('color','')}, {d.get('size','')}) — ${d.get('price',0)}"
                           for d in drafts]
            return (summary + f"\n\n**{len(drafts)} items in cart:**\n" + "\n".join(draft_lines) +
                    f"\n\n**Total: ${total:.2f}**\n\nAnything else or ready to pay?")
        return summary + f" **Total: ${total:.2f}**\n\nHow would you like to pay — **card** or **cash**?"
    if name == "confirm_payment":
        return (f"💳 Payment confirmed ({args.get('payment_method')}).\n\n"
                f"Now I need your delivery details: **full name, phone number, city, post office number**.")
    if name == "add_delivery_details":
        d = args
        return (f"📦 Delivery saved:\n• {d.get('name','—')} — {d.get('phone','—')}\n"
                f"• {d.get('city','—')}, post office #{d.get('post_office','—')}\n\n"
                f"Please confirm your order.")
    if name == "confirm_order":
        if result.get("error") == "no_drafts":
            return "⚠️ No items in cart yet."
        if "error" in result:
            return f"⚠️ {result['error']}"
        orders = result.get("confirmed_orders", [])
        if not orders:
            return "⚠️ No items confirmed."
        if len(orders) == 1:
            return f"🎉 Order **#{orders[0]['order_id']}** confirmed. Delivery in 2–3 days. Thank you!"
        ids = ", ".join(f"#{o['order_id']}" for o in orders)
        return f"🎉 **{len(orders)} orders confirmed:** {ids}. Thank you!"
    if name == "track_order":
        if "error" in result:
            return f"⚠️ {result['error']}"
        return (f"📍 **Order #{result['order_id']}** — `{result['status']}`\n"
                f"• {result.get('product','')} ({result.get('color','—')}, {result.get('size','—')})\n"
                f"• ETA: {result.get('estimated_delivery','—')}")
    if name == "cancel_order":
        if not result.get("cancelled"):
            return f"⚠️ {result.get('error') or result.get('message','Failed')}"
        return f"❎ Order **#{result['order_id']}** cancelled. Refund: **${result.get('refund_amount',0)}**."
    if name == "update_order_item":
        if not result.get("updated"):
            return f"⚠️ {result.get('error') or result.get('message','Failed')}"
        return f"✏️ Order **#{result['order_id']}** — {result['field']} → **{result['new_value']}**."
    if name == "create_return_request":
        if result.get("status") == "rejected":
            return f"⚠️ {result.get('error','Return rejected')}"
        return (f"↩️ Return **{result['return_id']}** approved. "
                f"Refund **${result.get('refund_amount',0)}**.\n_{result.get('instructions','')}_")
    if name == "get_current_order_context":
        active = result.get("active_orders", [])
        if not active:
            return "No active orders right now."
        lines_ = ["**Active orders:**"]
        for o in active:
            lines_.append(f"• #{o['order_id']} — {o.get('product','')} ({o.get('color','—')}, {o.get('size','—')}) — `{o.get('status','—')}`")
        return "\n".join(lines_)
    if name == "get_order_history":
        orders = result.get("orders", [])
        if not orders:
            return "No previous orders found."
        lines_ = ["**Order history:**"]
        for o in orders:
            lines_.append(f"• #{o['order_id']} — {o.get('product','')} ({o.get('color','—')}, {o.get('size','—')}) — `{o.get('status','—')}` — {o.get('date','—')}")
        return "\n".join(lines_)
    return f"```json\n{json.dumps(result, indent=2)}\n```"


def smart_inject_order(name, args, user_message, session):
    if name not in ("track_order", "cancel_order", "update_order_item", "create_return_request"):
        return args
    if args.get("order_id"):
        return args
    m = re.search(r"#?(\d{3,5})", user_message)
    if m:
        try:
            args["order_id"] = int(m.group(1))
            return args
        except ValueError:
            pass
    cid = session.get("customer_id")
    if cid is not None:
        for o in get_customer_orders(cid):
            if o.get("status") not in ("delivered", "cancelled", "returned"):
                args["order_id"] = o["order_id"]
                return args
    return args


def _generate(model, tokenizer, messages):
    ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            input_ids=ids,
            max_new_tokens=200,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    n_tok = out.shape[1] - ids.shape[1]
    text  = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()
    return text, n_tok


def _execute_function(name, args, session, msg, tool_log):
    args = smart_inject_order(name, args, msg, session)
    if name in FUNCTIONS:
        try:
            result = FUNCTIONS[name](session, **args)
        except Exception as e:
            result = {"error": f"{type(e).__name__}: {e}"}
    else:
        result = {"error": f"Unknown function: {name}"}
    tool_log.append(f"→ {name}({json.dumps(args, ensure_ascii=False)})")
    tool_log.append(f"← {json.dumps(result, ensure_ascii=False)}")
    if name == "check_product_availability" and result.get("available"):
        session["last_checked"] = {
            "product_name": args.get("product_name"),
            "color":        args.get("color"),
            "size":         args.get("size"),
        }
    if name in ("create_order", "place_order") and "error" not in result:
        session.pop("last_checked", None)
    return result


def run_model(model_label, msg, customer_name, model_hist, display_hist, use_formatted=False, use_state_machine=True):
    t0 = time.time()
    model, tokenizer = load_model(model_label)
    FastLanguageModel.for_inference(model)

    cust = find_customer_by_name(customer_name) if customer_name and customer_name != "New Customer" else None
    SESSION[model_label]["customer_id"] = cust["customer_id"] if cust else None
    session = SESSION[model_label]

    sys_prompt, mode_label = pick_system_prompt(msg, model_hist, model_label=model_label)
    working_hist = model_hist + [{"role": "user", "content": msg}]

    sm_mark   = "🔀 SM on" if use_state_machine else "🚫 SM off"
    tool_log  = [f"[{mode_label}] | {sm_mark} | {'📋 formatted' if use_formatted else '🤖 model response'}"]
    fn_first  = None
    total_tok = 0
    final_text = None

    sm = state_machine_call(msg, session) if (use_state_machine and mode_label == "🛒 shopping") else None
    if sm:
        name, args = sm
        tool_log.append(f"🔀 state_machine → {name}")
        result = _execute_function(name, args, session, msg, tool_log)
        fn_first = name
        tc_json = json.dumps({"name": name, "arguments": args}, ensure_ascii=False)
        working_hist.append({"role": "assistant", "content": f"<tool_call>{tc_json}</tool_call>"})
        working_hist.append({"role": "user",      "content": json.dumps(result, ensure_ascii=False)})
        final_text = format_tool_result(name, args, result, session=session)
        working_hist.append({"role": "assistant", "content": final_text})
        total_tok = len(final_text.split())
    else:
        MAX_STEPS = 8
        for _step in range(MAX_STEPS):
            messages = [{"role": "system", "content": sys_prompt}] + working_hist
            response, n_tok = _generate(model, tokenizer, messages)
            total_tok += n_tok

            if mode_label == "💬 support" and "<tool_call>" in response:
                tool_log.append("⛔ tool_call blocked (support mode)")
                response = strip_tool_call(response) or "Hello! How can I help you today?"

            if "<tool_call>" not in response:
                final_text = clean_response(response) or "I'm sorry, I couldn't form a response."
                working_hist.append({"role": "assistant", "content": response})
                break

            parsed = parse_tool_call(response)
            if not parsed:
                tool_log.append(f"⚠️ malformed: {response[:80]!r}")
                final_text = strip_tool_call(response) or "I had trouble parsing that response."
                working_hist.append({"role": "assistant", "content": response})
                break

            name, args = parsed
            if fn_first is None:
                fn_first = name
            result = _execute_function(name, args, session, msg, tool_log)

            tc_json = json.dumps({"name": name, "arguments": args}, ensure_ascii=False)
            working_hist.append({"role": "assistant", "content": f"<tool_call>{tc_json}</tool_call>"})
            working_hist.append({"role": "user",      "content": json.dumps(result, ensure_ascii=False)})

            if use_formatted:
                final_text = format_tool_result(name, args, result, session=session)
                working_hist.append({"role": "assistant", "content": final_text})
                break
        else:
            final_text = final_text or "I'm sorry, something went wrong."

    new_model_hist   = working_hist
    new_display_hist = display_hist + [
        {"role": "user",      "content": msg},
        {"role": "assistant", "content": final_text},
    ]

    latency = round(time.time() - t0, 2)
    return (final_text, tool_log, latency, total_tok,
            fn_first, mode_label, new_model_hist, new_display_hist)


def run_both(msg, customer_name, model_hist_q, model_hist_l, display_hist_q, display_hist_l, use_formatted=False, use_state_machine=True):
    def pack(t):
        text, log, latency, toks, fn, mode, mhist, dhist = t
        return {"text": text, "log": log, "latency": latency,
                "tokens": toks, "fn": fn, "mode": mode,
                "speed": round(toks / latency, 1) if latency else 0.0,
                "model_hist": mhist, "display_hist": dhist}
    q = pack(run_model("Qwen 2.5 1.5B", msg, customer_name, model_hist_q, display_hist_q, use_formatted, use_state_machine))
    l = pack(run_model("Llama 3.2 1B",  msg, customer_name, model_hist_l, display_hist_l, use_formatted, use_state_machine))
    return q, l

print("Chat engine ready.")


In [ ]:
import gradio as gr

LIGHT_CSS = """
:root, .gradio-container {
    --body-background-fill: #faf6f0 !important;
    --background-fill-primary: #ffffff !important;
    --background-fill-secondary: #f5efe6 !important;
    --color-accent: #8a6f4d !important;
    --color-accent-soft: #ead7bd !important;
    --button-primary-background-fill: #8a6f4d !important;
    --button-primary-background-fill-hover: #6f593d !important;
    --button-primary-text-color: #ffffff !important;
    --border-color-primary: #d9cfbe !important;
    --body-text-color: #666666 !important;
    font-family: 'Inter', system-ui, sans-serif !important;
}
.gradio-container { background: #faf6f0 !important; }
.model-card { background: #ffffff; border: 1px solid #e7decd; border-radius: 14px; padding: 14px; margin-bottom: 8px; }
.model-card h3 { margin: 0 0 8px 0; color: #ffffff !important; font-weight: 600; background: #8a6f4d; padding: 6px 12px; border-radius: 8px; display: inline-block; }
.metrics { display: flex; gap: 10px; flex-wrap: wrap; background: #f5efe6; border-radius: 10px; padding: 10px 12px; font-size: 13px; color: #666666; }
.metric-pill { background: #ffffff; border: 1px solid #e7decd; padding: 4px 10px; border-radius: 999px; }
.example-btn button { background: #ffffff !important; border: 1px solid #d9cfbe !important; color: #666666 !important; font-weight: 400 !important; }
.example-btn button:hover { background: #f5efe6 !important; }
/* white text only in text inputs and textareas (not dropdown popups) */
textarea, .block textarea { color: #ffffff !important; }
input[type=text]:not(.search-container input) { color: #ffffff !important; }
/* dropdown selected value shown in the input box — white; options list — dark */
.wrap-inner input { color: #ffffff !important; }
ul.options li, .choices__item { color: #3a3a3a !important; }
/* vertically center send buttons relative to the input field */
.send-row { align-items: center !important; }
.send-row > * { align-self: center !important; }
"""

DOTS_HTML = "<div style='color:#8a8074;font-style:italic;padding:6px 0'>● ● ● typing</div>"

def metrics_html(pack, name):
    if not pack:
        return f"<div class='metrics'><span class='metric-pill'><b>{name}</b></span></div>"
    fn = pack.get("fn") or "—"
    return ("<div class='metrics'>"
            f"<span class='metric-pill'><b>{name}</b></span>"
            f"<span class='metric-pill'>⏱ {pack['latency']}s</span>"
            f"<span class='metric-pill'>⚡ {pack['speed']} tok/s</span>"
            f"<span class='metric-pill'>📦 {pack['tokens']} tokens</span>"
            f"<span class='metric-pill'>🔧 {fn}</span>"
            "</div>")

EXAMPLES = {
    "👋 General":             [
        "Hello!",
        "Hi, I need help",
        "Good morning",
    ],
    "🔍 Search":              [
        "What dresses do you have?",
        "Show me tops in Black",
        "Browse your jeans collection",
        "What shoes do you have in White?",
    ],
    "🛒 Simple order":        [
        "I want to order Luna dress in Black size M",
        "I'd like Vega top in White size S",
        "Can I get Classic jeans in Blue size 30?",
    ],
    "✅ Confirm order":       [
        "Yes, place the order",
        "Go ahead",
        "Confirm",
        "I want to pay by card",
    ],
    "📦 Delivery details":    [
        "Anna Smith, +380501234567, Kyiv, 12",
        "Maria Garcia +380671234567 Lviv post office 5",
        "John Doe, +380991234567, Kharkiv, 3",
    ],
    "📍 Track order":         [
        "Where is my order?",
        "Track order #1010",
        "What's the status of my order 1031?",
    ],
    "✏️ Update":              [
        "Change size of order 1031 to 30",
        "I want to change the color of my order 1010 to Mint",
    ],
    "❌ Cancel":              [
        "Cancel my order",
        "Please cancel order 1031",
    ],
    "↩️ Return":              [
        "I want to return my order",
        "I'd like to return order 1011, doesn't fit",
    ],
    "📜 History":             [
        "Show me my orders",
        "What have I ordered before?",
    ],
}


def chat_step(msg, customer, mhist_q, mhist_l, dhist_q, dhist_l, log_q, log_l, response_mode, use_sm):
    use_formatted = (response_mode == "Formatted (template)")

    if not mhist_q:
        reset_session("Qwen 2.5 1.5B", None if customer == "New Customer" else customer)
    if not mhist_l:
        reset_session("Llama 3.2 1B", None if customer == "New Customer" else customer)

    if not msg.strip():
        yield ("", dhist_q, dhist_l,
               metrics_html(None, "Qwen 2.5 1.5B"), metrics_html(None, "Llama 3.2 1B"),
               log_q, log_l, mhist_q, mhist_l, dhist_q, dhist_l, get_db_view(customer))
        return

    typing_q = dhist_q + [{"role": "user", "content": msg}, {"role": "assistant", "content": DOTS_HTML}]
    typing_l = dhist_l + [{"role": "user", "content": msg}, {"role": "assistant", "content": DOTS_HTML}]
    yield ("", typing_q, typing_l,
           metrics_html(None, "Qwen 2.5 1.5B (computing…)"),
           metrics_html(None, "Llama 3.2 1B (computing…)"),
           log_q, log_l, mhist_q, mhist_l, dhist_q, dhist_l, get_db_view(customer))

    q, l = run_both(msg, customer, mhist_q, mhist_l, dhist_q, dhist_l, use_formatted=use_formatted, use_state_machine=use_sm)

    new_log_q = ("\n".join(q["log"]) + "\n\n" + (log_q or "")) if q["log"] else (log_q or "")
    new_log_l = ("\n".join(l["log"]) + "\n\n" + (log_l or "")) if l["log"] else (log_l or "")

    yield ("", q["display_hist"], l["display_hist"],
           metrics_html(q, "Qwen 2.5 1.5B"),
           metrics_html(l, "Llama 3.2 1B"),
           new_log_q, new_log_l,
           q["model_hist"], l["model_hist"],
           q["display_hist"], l["display_hist"],
           get_db_view(customer))


def _single_step(model_label, msg, customer, mhist, dhist, log, response_mode, use_sm):
    use_formatted = (response_mode == "Formatted (template)")
    name_short = model_label

    if not mhist:
        reset_session(model_label, None if customer == "New Customer" else customer)

    if not msg.strip():
        yield ("", dhist, metrics_html(None, name_short), log, mhist, dhist, get_db_view(customer))
        return

    typing = dhist + [{"role": "user", "content": msg}, {"role": "assistant", "content": DOTS_HTML}]
    yield ("", typing, metrics_html(None, f"{name_short} (computing…)"), log, mhist, dhist, get_db_view(customer))

    def pack(t):
        text, tlog, latency, toks, fn, mode, mh, dh = t
        return {"text": text, "log": tlog, "latency": latency,
                "tokens": toks, "fn": fn, "mode": mode,
                "speed": round(toks / latency, 1) if latency else 0.0,
                "model_hist": mh, "display_hist": dh}

    r = pack(run_model(model_label, msg, customer, mhist, dhist, use_formatted=use_formatted, use_state_machine=use_sm))
    new_log = ("\n".join(r["log"]) + "\n\n" + (log or "")) if r["log"] else (log or "")
    yield ("", r["display_hist"], metrics_html(r, name_short), new_log,
           r["model_hist"], r["display_hist"], get_db_view(customer))

def chat_step_q(msg, customer, mhist, dhist, log, response_mode, use_sm):
    yield from _single_step("Qwen 2.5 1.5B", msg, customer, mhist, dhist, log, response_mode, use_sm)

def chat_step_l(msg, customer, mhist, dhist, log, response_mode, use_sm):
    yield from _single_step("Llama 3.2 1B",  msg, customer, mhist, dhist, log, response_mode, use_sm)


with gr.Blocks(title="E-Commerce Chatbot — Model Comparison",
               theme=gr.themes.Soft(primary_hue="orange", neutral_hue="stone"),
               css=LIGHT_CSS) as demo:

    gr.Markdown("## E-Commerce Chatbot — Model Comparison Demo")
    gr.Markdown("_Side-by-side comparison of two fine-tuned models. Use the shared input to send the same message to both, or individual inputs below each chat to send different messages._")

    with gr.Row():
        customer_selector = gr.Dropdown(
            choices=["New Customer"] + get_customer_names(),
            value="New Customer",
            label="👤 Customer profile (shared by both models)",
            scale=3,
        )
        response_mode_toggle = gr.Radio(
            choices=["Model response (raw)", "Formatted (template)"],
            value="Model response (raw)",
            label="🔀 Response mode",
            info="Switching affects the next message only — no resend",
            scale=2,
        )
        state_machine_toggle = gr.Checkbox(
            value=True,
            label="🔀 State machine",
            info="Перехоплює критичні переходи (create_order, confirm_payment, add_delivery_details, confirm_order). Вимкніть, щоб побачити чисту поведінку моделі.",
            scale=1,
        )

    with gr.Row():
        with gr.Column():
            with gr.Group(elem_classes="model-card"):
                gr.Markdown("### Qwen 2.5 1.5B")
                metrics_q = gr.HTML(metrics_html(None, "Qwen 2.5 1.5B"))
                chat_q    = gr.Chatbot(type="messages", height=460, show_label=False, allow_tags=False)
                log_q_box = gr.Textbox(label="Function calls", lines=4, interactive=False, value="")
                with gr.Row(elem_classes="send-row"):
                    msg_q  = gr.Textbox(placeholder="Message to Qwen only…", show_label=False, scale=5)
                    send_q = gr.Button("Send →", scale=1)
        with gr.Column():
            with gr.Group(elem_classes="model-card"):
                gr.Markdown("### Llama 3.2 1B")
                metrics_l = gr.HTML(metrics_html(None, "Llama 3.2 1B"))
                chat_l    = gr.Chatbot(type="messages", height=460, show_label=False, allow_tags=False)
                log_l_box = gr.Textbox(label="Function calls", lines=4, interactive=False, value="")
                with gr.Row(elem_classes="send-row"):
                    msg_l  = gr.Textbox(placeholder="Message to Llama only…", show_label=False, scale=5)
                    send_l = gr.Button("Send →", scale=1)

    gr.Markdown("**Send to both models:**")
    with gr.Row(elem_classes="send-row"):
        msg_input = gr.Textbox(placeholder="Type once — both models reply.", show_label=False, scale=8, autofocus=True)
        send_btn  = gr.Button("Send both", variant="primary", scale=1)
        clear_btn = gr.Button("Clear", scale=1)

    gr.Markdown("### Sample prompts")
    for label, qs in EXAMPLES.items():
        with gr.Accordion(label, open=False):
            with gr.Row():
                for q in qs:
                    btn = gr.Button(q, size="sm", elem_classes="example-btn")
                    btn.click(fn=lambda v=q: v, outputs=msg_input)

    gr.Markdown("### Database snapshot")
    db_view = gr.HTML(get_db_view())

    model_hist_q_state   = gr.State([])
    model_hist_l_state   = gr.State([])
    display_hist_q_state = gr.State([])
    display_hist_l_state = gr.State([])

    shared_outputs = [
        msg_input,
        chat_q, chat_l,
        metrics_q, metrics_l,
        log_q_box, log_l_box,
        model_hist_q_state, model_hist_l_state,
        display_hist_q_state, display_hist_l_state,
        db_view,
    ]
    shared_inputs = [
        msg_input, customer_selector,
        model_hist_q_state, model_hist_l_state,
        display_hist_q_state, display_hist_l_state,
        log_q_box, log_l_box,
        response_mode_toggle, state_machine_toggle,
    ]
    send_btn.click(chat_step,   inputs=shared_inputs, outputs=shared_outputs)
    msg_input.submit(chat_step, inputs=shared_inputs, outputs=shared_outputs)

    single_q_outputs = [msg_q, chat_q, metrics_q, log_q_box,
                        model_hist_q_state, display_hist_q_state, db_view]
    single_q_inputs  = [msg_q, customer_selector,
                        model_hist_q_state, display_hist_q_state,
                        log_q_box, response_mode_toggle, state_machine_toggle]
    send_q.click(chat_step_q,  inputs=single_q_inputs, outputs=single_q_outputs)
    msg_q.submit(chat_step_q,  inputs=single_q_inputs, outputs=single_q_outputs)

    single_l_outputs = [msg_l, chat_l, metrics_l, log_l_box,
                        model_hist_l_state, display_hist_l_state, db_view]
    single_l_inputs  = [msg_l, customer_selector,
                        model_hist_l_state, display_hist_l_state,
                        log_l_box, response_mode_toggle, state_machine_toggle]
    send_l.click(chat_step_l,  inputs=single_l_inputs, outputs=single_l_outputs)
    msg_l.submit(chat_step_l,  inputs=single_l_inputs, outputs=single_l_outputs)

    def _clear(customer):
        reset_session("Qwen 2.5 1.5B", None if customer == "New Customer" else customer)
        reset_session("Llama 3.2 1B",  None if customer == "New Customer" else customer)
        return ("", [], [],
                metrics_html(None, "Qwen 2.5 1.5B"),
                metrics_html(None, "Llama 3.2 1B"),
                "", "",
                [], [], [], [],
                get_db_view(customer))

    clear_btn.click(_clear, inputs=[customer_selector], outputs=shared_outputs)
    customer_selector.change(lambda c: get_db_view(c), inputs=[customer_selector], outputs=[db_view])


print("Preloading Qwen…")
load_model("Qwen 2.5 1.5B")
print("Preloading Llama…")
load_model("Llama 3.2 1B")

demo.launch(share=True)